In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

raw_dir = ROOT / "data" / "raw" / "SWOW-EN18"

print("Files in", raw_dir, "\n")
for f in sorted(raw_dir.iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        # peek headers only — read 0 rows
        try:
            cols = pd.read_csv(f, nrows=0, sep=None, engine="python").columns.tolist()
        except Exception as e:
            cols = [f"(couldn't read: {e})"]
        print(f"{f.name}  —  {size_mb:.1f} MB")
        print("   cols:", cols, "\n")

Files in /Users/mac/Desktop/semantic_graph_project/data/raw/SWOW-EN18 

SWOW-EN.R100.20180827.csv  —  144.4 MB
   cols: ['Unnamed: 0', 'id', 'participantID', 'age', 'gender', 'nativeLanguage', 'country', 'education', 'created_at', 'cue', 'R1', 'R2', 'R3'] 

SWOW-EN.complete.20180827.csv  —  232.4 MB
   cols: ['Unnamed: 0', 'id', 'participantID', 'created_at', 'age', 'nativeLanguage', 'gender', 'education', 'city', 'country', 'section', 'cue', 'R1Raw', 'R2Raw', 'R3Raw', 'R1', 'R2', 'R3'] 

cueStats.SWOW-EN.R1.20180827.csv  —  0.6 MB
   cols: ['Unnamed: 0', 'cue', 'coverage', 'H', 'unknown', 'N'] 

cueStats.SWOW-EN.R123.20180827.csv  —  0.7 MB
   cols: ['Unnamed: 0', 'cue', 'coverage', 'H', 'unknown', 'xR2', 'xR3', 'N'] 

responseStats.SWOW-EN.20180827.csv  —  4.2 MB
   cols: ['Unnamed: 0', 'response', 'Freq.R1', 'Types.R1', 'Freq.R123', 'Types.R123'] 

strength.SWOW-EN.R1.20180827.csv  —  17.1 MB
   cols: ['cue', 'response', 'R1', 'N', 'R1.Strength'] 

strength.SWOW-EN.R123.20180827.csv

In [3]:
import pandas as pd

complete_path = raw_dir / "SWOW-EN.complete.20180827.csv"

# read only what we need — keeps memory low on the 232MB file
df_meta = pd.read_csv(
    complete_path,
    usecols=["participantID", "nativeLanguage", "cue", "R1"],
)

print("total response rows:", len(df_meta))
print("unique participants:", df_meta["participantID"].nunique(), "\n")

# distribution of nativeLanguage — top 20
vc = df_meta["nativeLanguage"].value_counts(dropna=False)
print("nativeLanguage — top 20 by response count:")
print(vc.head(20), "\n")
print("distinct nativeLanguage values:", df_meta["nativeLanguage"].nunique(dropna=False))
print("rows with missing nativeLanguage:", df_meta["nativeLanguage"].isna().sum())

# participant-level (not response-level) counts
ppl = df_meta.drop_duplicates("participantID")["nativeLanguage"].value_counts(dropna=False)
print("\nnativeLanguage — top 20 by PARTICIPANT count:")
print(ppl.head(20))

total response rows: 1356362
unique participants: 88722 

nativeLanguage — top 20 by response count:
nativeLanguage
United States        674224
United Kingdom       177892
Canada               139778
Australia             70070
Other_Foreign         50070
German                24260
Spanish               23246
Dutch_Netherlands     15535
Dutch_Flanders        15183
New Zealand           12890
French                12078
Ireland               11371
Italian                9717
Finnish                9665
Russian                8571
Swedish                7541
Portuguese             7473
South Africa           7143
Mandarin               6743
Singapore              6479
Name: count, dtype: int64 

distinct nativeLanguage values: 60
rows with missing nativeLanguage: 0

nativeLanguage — top 20 by PARTICIPANT count:
nativeLanguage
United States        44659
United Kingdom       11299
Canada                9486
Australia             4557
Other_Foreign         3150
German                1562
S

In [4]:
import pandas as pd

complete_path = raw_dir / "SWOW-EN.complete.20180827.csv"

df_chk = pd.read_csv(
    complete_path,
    usecols=["participantID", "nativeLanguage", "country"],
).drop_duplicates("participantID")

print("participants:", len(df_chk), "\n")

# Are they the same field or different?
print("distinct nativeLanguage:", df_chk["nativeLanguage"].nunique())
print("distinct country:", df_chk["country"].nunique(), "\n")

# Cross-tab: for a few language-named values, what countries appear?
for lang in ["German", "Spanish", "United States"]:
    sub = df_chk[df_chk["nativeLanguage"] == lang]
    print(f"nativeLanguage = {lang!r}  (n={len(sub)})")
    print("  top countries:", sub["country"].value_counts().head(5).to_dict(), "\n")

participants: 88722 

distinct nativeLanguage: 60
distinct country: 320 

nativeLanguage = 'German'  (n=1562)
  top countries: {'Germany': 833, 'United Kingdom': 126, 'United States': 121, 'Austria': 99, 'Switzerland': 81} 

nativeLanguage = 'Spanish'  (n=1456)
  top countries: {'Spain': 330, 'United States': 310, 'Argentina': 142, 'Mexico': 102, 'Uruguay': 72} 

nativeLanguage = 'United States'  (n=44659)
  top countries: {'United States': 41757, 'United Kingdom': 313, 'notfound': 232, 'Canada': 214, 'Germany': 172} 



In [5]:
import pandas as pd

complete_path = raw_dir / "SWOW-EN.complete.20180827.csv"

df_lang = pd.read_csv(
    complete_path,
    usecols=["participantID", "nativeLanguage", "cue", "R1", "R2", "R3"],
)

NATIVE_ENGLISH = {
    "United States", "United Kingdom", "Canada", "Australia",
    "New Zealand", "Ireland", "South Africa", "Singapore"
}

df_lang["is_native"] = df_lang["nativeLanguage"].isin(NATIVE_ENGLISH)

# participant-level counts
ppl = df_lang.drop_duplicates("participantID")
print("participants total:", len(ppl))
print("native:", ppl["is_native"].sum())
print("non-native:", (~ppl["is_native"]).sum())

# response-level counts (this is what we'll actually use for distributions)
print("\nresponse rows total:", len(df_lang))
print("native responses:", df_lang["is_native"].sum())
print("non-native responses:", (~df_lang["is_native"]).sum())

participants total: 88722
native: 72429
non-native: 16293

response rows total: 1356362
native responses: 1099847
non-native responses: 256515


In [7]:
import pandas as pd

# Reuse df_lang from your last cell (has is_native, cue, R1/R2/R3).
# Reshape to long: one row per (cue, response, group), pooling R1+R2+R3 to match the R123 graph.
long = df_lang.melt(
    id_vars=["cue", "is_native"],
    value_vars=["R1", "R2", "R3"],
    value_name="response",
).drop(columns="variable")

# drop missing/empty responses (SWOW uses NA and "No more responses" style blanks)
long = long[long["response"].notna()]
long = long[long["response"].astype(str).str.strip() != ""]

def build_dist(df_group):
    # count (cue -> response) pairs, then normalize within each cue to a probability
    counts = df_group.groupby(["cue", "response"]).size().rename("n").reset_index()
    counts["p"] = counts["n"] / counts.groupby("cue")["n"].transform("sum")
    return counts

nat  = build_dist(long[long["is_native"]])
nnat = build_dist(long[~long["is_native"]])

print("NATIVE     — cues:", nat["cue"].nunique(),  "| cue-response pairs:", len(nat))
print("NON-NATIVE — cues:", nnat["cue"].nunique(), "| cue-response pairs:", len(nnat))

# sparsity check: responses per cue in each group
nat_rpc  = nat.groupby("cue").size()
nnat_rpc = nnat.groupby("cue").size()
print("\nresponses-per-cue (distinct responses):")
print("  native     median:", int(nat_rpc.median()),  "| mean:", round(nat_rpc.mean(), 1))
print("  non-native median:", int(nnat_rpc.median()), "| mean:", round(nnat_rpc.mean(), 1))

# cue overlap — can we even compare them cue-for-cue?
shared = set(nat["cue"]) & set(nnat["cue"])
print("\ncues in both groups:", len(shared))
print("cues native-only:", len(set(nat['cue']) - set(nnat['cue'])))
print("cues non-native-only:", len(set(nnat['cue']) - set(nat['cue'])))

NATIVE     — cues: 12290 | cue-response pairs: 1302832
NON-NATIVE — cues: 12291 | cue-response pairs: 441971

responses-per-cue (distinct responses):
  native     median: 105 | mean: 106.0
  non-native median: 34 | mean: 36.0

cues in both groups: 12290
cues native-only: 0
cues non-native-only: 1


In [8]:
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

# Work from the long token-level tables (one row per response token), not the counted dists.
nat_long  = long[long["is_native"]][["cue", "response"]]
nnat_long = long[~long["is_native"]][["cue", "response"]]

# how many non-native tokens per cue — this is our per-cue target depth
target_n = nnat_long.groupby("cue").size()

def subsample_to_target(df_tokens, target_per_cue, rng):
    out = []
    for cue, grp in df_tokens.groupby("cue", sort=False):
        k = target_per_cue.get(cue, 0)
        if k == 0:
            continue
        if len(grp) <= k:
            out.append(grp)                      # already at/below target, keep all
        else:
            idx = rng.choice(len(grp), size=k, replace=False)
            out.append(grp.iloc[idx])
    return pd.concat(out, ignore_index=True)

nat_sub_long = subsample_to_target(nat_long, target_n, rng)

def build_dist(df_group):
    counts = df_group.groupby(["cue", "response"]).size().rename("n").reset_index()
    counts["p"] = counts["n"] / counts.groupby("cue")["n"].transform("sum")
    return counts

nat_sub = build_dist(nat_sub_long)   # depth-matched native ground truth
# nnat (non-native) already built in previous cell — unchanged

# verify the match
nat_rpc_sub = nat_sub.groupby("cue").size()
nnat_rpc    = nnat.groupby("cue").size()
print("after subsampling — responses-per-cue (distinct):")
print("  native (matched) median:", int(nat_rpc_sub.median()), "| mean:", round(nat_rpc_sub.mean(), 1))
print("  non-native        median:", int(nnat_rpc.median()),   "| mean:", round(nnat_rpc.mean(), 1))

print("\ntoken totals:")
print("  native matched tokens:", len(nat_sub_long), "| non-native tokens:", len(nnat_long))
print("  native pairs:", len(nat_sub), "| non-native pairs:", len(nnat))

after subsampling — responses-per-cue (distinct):
  native (matched) median: 34 | mean: 36.1
  non-native        median: 34 | mean: 36.0

token totals:
  native matched tokens: 769492 | non-native tokens: 769534
  native pairs: 443412 | non-native pairs: 441971


In [9]:
import numpy as np

nnat_rpc = nnat.groupby("cue").size()  # non-native is the binding (thinner) side

print("non-native responses-per-cue — percentiles:")
for q in [1, 5, 10, 25, 50, 75, 90]:
    print(f"  {q:>2}th pct: {int(np.percentile(nnat_rpc, q))}")

print("\ncues surviving a minimum-depth floor:")
total = nnat_rpc.size
for floor in [10, 15, 20, 25]:
    keep = (nnat_rpc >= floor).sum()
    print(f"  >= {floor}: {keep} cues kept ({keep/total:.0%})")

non-native responses-per-cue — percentiles:
   1th pct: 15
   5th pct: 19
  10th pct: 22
  25th pct: 27
  50th pct: 34
  75th pct: 43
  90th pct: 53

cues surviving a minimum-depth floor:
  >= 10: 12279 cues kept (100%)
  >= 15: 12172 cues kept (99%)
  >= 20: 11635 cues kept (95%)
  >= 25: 10227 cues kept (83%)


In [11]:
import json

FLOOR = 15

nnat_rpc = nnat.groupby("cue").size()
keep_cues = set(nnat_rpc[nnat_rpc >= FLOOR].index)

nat_gt  = nat_sub[nat_sub["cue"].isin(keep_cues)].reset_index(drop=True)
nnat_gt = nnat[nnat["cue"].isin(keep_cues)].reset_index(drop=True)

print("floor:", FLOOR)
print("cues kept:", len(keep_cues))
print("native cues:", nat_gt["cue"].nunique(), "| non-native cues:", nnat_gt["cue"].nunique())
print("native pairs:", len(nat_gt), "| non-native pairs:", len(nnat_gt))

proc = ROOT / "data" / "processed"
proc.mkdir(parents=True, exist_ok=True)

nat_gt.to_pickle(proc / "L2_groundtruth_native.pkl")
nnat_gt.to_pickle(proc / "L2_groundtruth_nonnative.pkl")

stamp = {
    "seed": SEED,
    "floor": FLOOR,
    "n_cues": len(keep_cues),
    "native_pairs": len(nat_gt),
    "nonnative_pairs": len(nnat_gt),
    "native_tokens": int(len(nat_sub_long)),
    "nonnative_tokens": int(len(nnat_long)),
}
(proc / "L2_groundtruth_stamp.json").write_text(json.dumps(stamp, indent=2))
print("\ncached + stamped:", stamp)

floor: 15
cues kept: 12172
native cues: 12172 | non-native cues: 12172
native pairs: 441331 | non-native pairs: 440502

cached + stamped: {'seed': 42, 'floor': 15, 'n_cues': 12172, 'native_pairs': 441331, 'nonnative_pairs': 440502, 'native_tokens': 769492, 'nonnative_tokens': 769534}


In [12]:
import pickle
import numpy as np
from collections import Counter

# load the English graph (directed, edge attr R123.Strength)
with open(ROOT / "data" / "processed" / "swow_en_graph.pkl", "rb") as f:
    G = pickle.load(f)

print("graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")

all_nodes = np.array(list(G.nodes()))
rng = np.random.default_rng(0)

# precompute neighbors + normalized weights for fast sampling from one cue
def neighbor_table(cue):
    nbrs, wts = [], []
    for _, tgt, data in G.out_edges(cue, data=True):
        nbrs.append(tgt)
        wts.append(data.get("R123.Strength", 0.0))
    wts = np.array(wts, dtype=float)
    if wts.sum() == 0:
        return None, None
    return np.array(nbrs), wts / wts.sum()

def walk_once(cue, eps, nbrs, probs, rng):
    # with prob eps: random jump; else follow a weighted edge
    if nbrs is None or rng.random() < eps:
        return all_nodes[rng.integers(len(all_nodes))]
    return nbrs[rng.integers(len(nbrs))] if False else rng.choice(nbrs, p=probs)

def simulate(cue, eps, n=5000):
    nbrs, probs = neighbor_table(cue)
    if nbrs is None:
        return None
    c = Counter(walk_once(cue, eps, nbrs, probs, rng) for _ in range(n))
    return c

CUE = "dog"
for eps in [0.0, 0.5]:
    c = simulate(CUE, eps, n=5000)
    print(f"\nepsilon={eps} — top 10 responses for '{CUE}':")
    for word, cnt in c.most_common(10):
        print(f"  {word:15s} {cnt/5000:.3f}")

graph: 24657 nodes, 279410 edges

epsilon=0.0 — top 10 responses for 'dog':


AttributeError: 'NoneType' object has no attribute 'most_common'

In [13]:
# is "dog" even in the graph?
print("'dog' in graph:", "dog" in G)
if "dog" in G:
    print("'dog' out-degree:", G.out_degree("dog"))

# show a few real nodes + their out-degrees so we pick a valid cue
sample = list(G.nodes())[:10]
print("\nsample nodes:", sample)

# find some cues that definitely have out-edges
good = [n for n in G.nodes() if G.out_degree(n) > 5][:10]
print("\ncues with >5 out-edges:", good)

'dog' in graph: True
'dog' out-degree: 0

sample nodes: ['a', 'abc', 'alpha', 'alphabet', 'an', 'and', 'apple', 'article', 'at', 'bat']

cues with >5 out-edges: ['a', 'alpha', 'alphabet', 'an', 'and', 'apple', 'article', 'at', 'bat', 'beginning']


In [14]:
print("dog out-degree:", G.out_degree("dog"))
print("dog in-degree :", G.in_degree("dog"))

# peek at what dog connects to each way
print("\ndog OUT edges (cue->?):", list(G.out_edges("dog", data=True))[:5])
print("dog IN edges (?->dog) :", list(G.in_edges("dog", data=True))[:5])

dog out-degree: 0
dog in-degree : 358

dog OUT edges (cue->?): []
dog IN edges (?->dog) : [('a', 'dog', {'R123.Strength': 0.011111111111111112}), ('abandon', 'dog', {'R123.Strength': 0.010752688172043012}), ('adopt', 'dog', {'R123.Strength': 0.025925925925925925}), ('adorable', 'dog', {'R123.Strength': 0.013793103448275862}), ('adore', 'dog', {'R123.Strength': 0.01048951048951049})]


In [15]:
import pandas as pd

s = pd.read_csv(ROOT / "data" / "raw" / "SWOW-EN18" / "strength.SWOW-EN.R123.20180827.csv")
print(s.columns.tolist())

# how is 'dog' represented in the raw strengths?
print("\nrows where cue == 'dog':", (s["cue"] == "dog").sum())
print("rows where response == 'dog':", (s["response"] == "dog").sum())

# show dog's actual responses from the RAW data
print("\nraw: cue='dog' top responses:")
print(s[s["cue"] == "dog"].sort_values("R123.Strength", ascending=False)
        .head(8)[["cue", "response", "R123.Strength"]].to_string(index=False))

ParserError: Error tokenizing data. C error: Expected 1 fields in line 114, saw 2


In [16]:
import pandas as pd

s = pd.read_csv(ROOT / "data" / "raw" / "SWOW-EN18" / "strength.SWOW-EN.R123.20180827.csv", sep="\t")
print(s.columns.tolist())

print("\nrows where cue == 'dog':", (s["cue"] == "dog").sum())
print("rows where response == 'dog':", (s["response"] == "dog").sum())

print("\nraw: cue='dog' top responses:")
print(s[s["cue"] == "dog"].sort_values("R123.Strength", ascending=False)
        .head(8)[["cue", "response", "R123.Strength"]].to_string(index=False))

['cue', 'response', 'R123', 'N', 'R123.Strength']

rows where cue == 'dog': 0
rows where response == 'dog': 899

raw: cue='dog' top responses:
Empty DataFrame
Columns: [cue, response, R123.Strength]
Index: []


In [17]:
# any cue that looks like 'dog' regardless of case/spacing?
mask = s["cue"].astype(str).str.strip().str.lower() == "dog"
print("cues matching 'dog' (case/space-insensitive):", mask.sum())
if mask.sum():
    print(s[mask]["cue"].unique()[:10])

# what cues START with 'dog'? (dog, dogs, doggy...)
dogcues = s[s["cue"].astype(str).str.lower().str.startswith("dog")]["cue"].unique()
print("\ncues starting with 'dog':", dogcues[:20])

# sanity: how many distinct cues total, and do common words exist as cues?
print("\ntotal distinct cues:", s["cue"].nunique())
for w in ["cat", "house", "water", "happy"]:
    print(f"  '{w}' is a cue:", (s["cue"] == w).any())

cues matching 'dog' (case/space-insensitive): 0

cues starting with 'dog': <StringArray>
[]
Length: 0, dtype: str

total distinct cues: 8660
  'cat' is a cue: True
  'house' is a cue: False
  'water' is a cue: True
  'happy' is a cue: False


In [18]:
# confirm cat is a graph cue with out-edges
print("cat out-degree:", G.out_degree("cat"))
print("cat top OUT edges:", sorted(G.out_edges("cat", data=True),
      key=lambda e: -e[2]["R123.Strength"])[:5])

cat out-degree: 39
cat top OUT edges: [('cat', 'dog', {'R123.Strength': 0.20469798657718122}), ('cat', 'feline', {'R123.Strength': 0.0738255033557047}), ('cat', 'meow', {'R123.Strength': 0.05704697986577181}), ('cat', 'purr', {'R123.Strength': 0.0436241610738255}), ('cat', 'mouse', {'R123.Strength': 0.040268456375838924})]


In [19]:
import numpy as np
from collections import Counter

all_nodes = np.array(list(G.nodes()))
rng = np.random.default_rng(0)

def neighbor_table(cue):
    nbrs, wts = [], []
    for _, tgt, data in G.out_edges(cue, data=True):
        nbrs.append(tgt)
        wts.append(data.get("R123.Strength", 0.0))
    if not nbrs:
        return None, None
    wts = np.array(wts, dtype=float)
    return np.array(nbrs), wts / wts.sum()

def simulate(cue, eps, n=5000):
    nbrs, probs = neighbor_table(cue)
    out = []
    for _ in range(n):
        if nbrs is None or rng.random() < eps:
            out.append(all_nodes[rng.integers(len(all_nodes))])
        else:
            out.append(rng.choice(nbrs, p=probs))
    return Counter(out)

CUE = "cat"
for eps in [0.0, 0.5]:
    c = simulate(CUE, eps, n=5000)
    print(f"\nepsilon={eps} — top 10 for '{CUE}':")
    for word, cnt in c.most_common(10):
        print(f"  {word:15s} {cnt/5000:.3f}")


epsilon=0.0 — top 10 for 'cat':
  dog             0.249
  feline          0.097
  meow            0.075
  purr            0.059
  mouse           0.052
  fur             0.039
  kitten          0.031
  soft            0.027
  animal          0.026
  pet             0.025

epsilon=0.5 — top 10 for 'cat':
  dog             0.124
  feline          0.045
  meow            0.035
  purr            0.026
  mouse           0.026
  fur             0.023
  soft            0.016
  kitten          0.015
  furry           0.014
  kitty           0.011


In [20]:
graph_cues = set(G.nodes())
gt_cues = set(nnat_gt["cue"])  # ground-truth cues (both groups share these)

usable = gt_cues & graph_cues
# but a cue is only usable if it has OUT-edges (responses) in the graph
usable_with_edges = {c for c in usable if G.out_degree(c) > 0}

print("ground-truth cues:", len(gt_cues))
print("graph cues (any):", len(graph_cues))
print("GT cues present in graph:", len(usable))
print("GT cues present AND with out-edges:", len(usable_with_edges))
print("\nlost: in GT but no walkable cue:", len(gt_cues - usable_with_edges))

ground-truth cues: 12172
graph cues (any): 24657
GT cues present in graph: 11557
GT cues present AND with out-edges: 8283

lost: in GT but no walkable cue: 3889


In [21]:
# how much response MASS (tokens), not just cue count, survives?
nnat_tokens_per_cue = nnat_long.groupby("cue").size()  # token counts per cue
nat_tokens_per_cue  = nat_sub_long.groupby("cue").size()

walkable = usable_with_edges

def mass_kept(tok_per_cue):
    total = tok_per_cue.sum()
    kept  = tok_per_cue[tok_per_cue.index.isin(walkable)].sum()
    return kept, total, kept/total

for name, tpc in [("non-native", nnat_tokens_per_cue), ("native", nat_tokens_per_cue)]:
    kept, total, frac = mass_kept(tpc)
    print(f"{name}: {kept:,} / {total:,} tokens kept ({frac:.1%})")

non-native: 514,265 / 769,495 tokens kept (66.8%)
native: 514,265 / 769,492 tokens kept (66.8%)


In [22]:
# are these really two different series, or did we double-count one?
print("non-native total tokens:", int(nnat_tokens_per_cue.sum()))
print("native total tokens    :", int(nat_tokens_per_cue.sum()))
print("same object?", nnat_tokens_per_cue is nat_tokens_per_cue)

# kept tokens computed independently, side by side
w = usable_with_edges
print("\nnnat kept:", int(nnat_tokens_per_cue[nnat_tokens_per_cue.index.isin(w)].sum()))
print("nat  kept:", int(nat_tokens_per_cue[nat_tokens_per_cue.index.isin(w)].sum()))

# how many cues contribute on each side?
print("\nnnat walkable cues:", nnat_tokens_per_cue.index.isin(w).sum())
print("nat  walkable cues:", nat_tokens_per_cue.index.isin(w).sum())

non-native total tokens: 769495
native total tokens    : 769492
same object? False

nnat kept: 514265
nat  kept: 514265

nnat walkable cues: 8283
nat  walkable cues: 8283


In [23]:
import numpy as np
from collections import Counter

def walk_dist(cue, eps, n=20000):
    """walk's response distribution as a dict word->prob"""
    nbrs, probs = neighbor_table(cue)
    if nbrs is None:
        return None
    c = Counter()
    for _ in range(n):
        if rng.random() < eps:
            c[all_nodes[rng.integers(len(all_nodes))]] += 1
        else:
            c[rng.choice(nbrs, p=probs)] += 1
    tot = sum(c.values())
    return {w: v / tot for w, v in c.items()}

def real_dist(cue, gt_df):
    """real data distribution for a cue as dict word->prob"""
    sub = gt_df[gt_df["cue"] == cue]
    return dict(zip(sub["response"], sub["p"]))

def jsd(p, q):
    """Jensen-Shannon divergence over union vocabulary, log base 2 -> [0,1]"""
    vocab = set(p) | set(q)
    pv = np.array([p.get(w, 0.0) for w in vocab])
    qv = np.array([q.get(w, 0.0) for w in vocab])
    pv = pv / pv.sum(); qv = qv / qv.sum()
    m = 0.5 * (pv + qv)
    def kl(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log2(a[mask] / b[mask]))
    return 0.5 * kl(pv, m) + 0.5 * kl(qv, m)

CUE = "cat"
real = real_dist(CUE, nat_gt)   # native ground truth for this cue
print(f"cue='{CUE}', real responses:", len(real))
print(f"{'eps':>5} {'JSD':>8}")
for eps in [0.0, 0.1, 0.2, 0.3, 0.5, 0.8]:
    w = walk_dist(CUE, eps)
    print(f"{eps:5.1f} {jsd(w, real):8.4f}")

cue='cat', real responses: 25
  eps      JSD
  0.0   0.3659
  0.1   0.3975
  0.2   0.4318
  0.3   0.4721
  0.5   0.5556
  0.8   0.7364


In [24]:
CUE = "cat"
real_nat  = real_dist(CUE, nat_gt)
real_nnat = real_dist(CUE, nnat_gt)
print(f"cue='{CUE}' | native responses: {len(real_nat)} | non-native: {len(real_nnat)}")
print(f"{'eps':>5} {'JSD_native':>11} {'JSD_nonnat':>11}")
for eps in [0.0, 0.1, 0.2, 0.3, 0.5, 0.8]:
    w = walk_dist(CUE, eps)              # one walk, scored against both
    print(f"{eps:5.1f} {jsd(w, real_nat):11.4f} {jsd(w, real_nnat):11.4f}")

cue='cat' | native responses: 25 | non-native: 22
  eps  JSD_native  JSD_nonnat
  0.0      0.3648      0.3227
  0.1      0.3973      0.3570
  0.2      0.4328      0.3940
  0.3      0.4707      0.4390
  0.5      0.5552      0.5325
  0.8      0.7412      0.7354


In [25]:
import numpy as np

# fixed reproducible sample of walkable cues
walkable_cues = sorted(usable_with_edges)
samp_rng = np.random.default_rng(7)
SAMPLE = list(samp_rng.choice(walkable_cues, size=200, replace=False))

EPS_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7]

# pre-index ground truth by cue for speed
nat_by_cue  = {c: g for c, g in nat_gt.groupby("cue")}
nnat_by_cue = {c: g for c, g in nnat_gt.groupby("cue")}

def real_dist_fast(cue, by_cue):
    sub = by_cue.get(cue)
    if sub is None: return None
    return dict(zip(sub["response"], sub["p"]))

results = {eps: {"nat": [], "nnat": []} for eps in EPS_GRID}

for i, cue in enumerate(SAMPLE):
    rn  = real_dist_fast(cue, nat_by_cue)
    rnn = real_dist_fast(cue, nnat_by_cue)
    if rn is None or rnn is None:
        continue
    for eps in EPS_GRID:
        w = walk_dist(cue, eps, n=8000)   # smaller n for speed across 200 cues
        if w is None: continue
        results[eps]["nat"].append(jsd(w, rn))
        results[eps]["nnat"].append(jsd(w, rnn))
    if (i+1) % 50 == 0:
        print(f"  ...{i+1}/200 cues done")

print(f"\n{'eps':>5} {'nat_mean':>9} {'nat_med':>8} {'nnat_mean':>10} {'nnat_med':>9}")
for eps in EPS_GRID:
    nm = np.mean(results[eps]["nat"]);  nmd = np.median(results[eps]["nat"])
    xm = np.mean(results[eps]["nnat"]); xmd = np.median(results[eps]["nnat"])
    print(f"{eps:5.1f} {nm:9.4f} {nmd:8.4f} {xm:10.4f} {xmd:9.4f}")

  ...50/200 cues done
  ...100/200 cues done
  ...150/200 cues done
  ...200/200 cues done

  eps  nat_mean  nat_med  nnat_mean  nnat_med
  0.0    0.3505   0.3411     0.4942    0.4835
  0.1    0.3831   0.3724     0.5184    0.5092
  0.2    0.4179   0.4071     0.5449    0.5360
  0.3    0.4559   0.4470     0.5735    0.5636
  0.4    0.4985   0.4887     0.6055    0.5954
  0.5    0.5454   0.5382     0.6414    0.6317
  0.7    0.6608   0.6535     0.7293    0.7236


In [27]:
import pandas as pd, numpy as np

rs = pd.read_csv(ROOT / "data" / "raw" / "SWOW-EN18" / "responseStats.SWOW-EN.20180827.csv")
print("cols:", rs.columns.tolist())

cols: ['Unnamed: 0', 'response', 'Freq.R1', 'Types.R1', 'Freq.R123', 'Types.R123']


In [28]:
node_set = set(G.nodes())
rs_f = rs[rs["response"].isin(node_set)][["response", "Freq.R123"]].dropna()
rs_f = rs_f[rs_f["Freq.R123"] > 0]

noise_words = rs_f["response"].to_numpy()
noise_w = rs_f["Freq.R123"].to_numpy(dtype=float)
noise_p = noise_w / noise_w.sum()

print("noise vocabulary size:", len(noise_words))
print("top noise-landing words:",
      [noise_words[i] for i in np.argsort(noise_w)[-8:][::-1]])

def walk_dist_fw(cue, eps, n=20000):
    nbrs, probs = neighbor_table(cue)
    if nbrs is None: return None
    c = Counter()
    for _ in range(n):
        if rng.random() < eps:
            c[noise_words[rng.choice(len(noise_words), p=noise_p)]] += 1
        else:
            c[rng.choice(nbrs, p=probs)] += 1
    tot = sum(c.values())
    return {w: v/tot for w, v in c.items()}

print("\n'cat' at eps=0.5 (freq-weighted noise) — top 10:")
d = walk_dist_fw("cat", 0.5)
for w, p in sorted(d.items(), key=lambda x: -x[1])[:10]:
    print(f"  {w:15s} {p:.3f}")

noise vocabulary size: 22750
top noise-landing words: ['money', 'water', 'food', 'car', 'music', 'green', 'red', 'love']

'cat' at eps=0.5 (freq-weighted noise) — top 10:
  dog             0.124
  feline          0.049
  meow            0.036
  purr            0.028
  mouse           0.025
  fur             0.021
  kitten          0.016
  soft            0.015
  animal          0.013
  furry           0.012


In [29]:
import numpy as np

results_fw = {eps: {"nat": [], "nnat": []} for eps in EPS_GRID}

for i, cue in enumerate(SAMPLE):
    rn  = real_dist_fast(cue, nat_by_cue)
    rnn = real_dist_fast(cue, nnat_by_cue)
    if rn is None or rnn is None:
        continue
    for eps in EPS_GRID:
        w = walk_dist_fw(cue, eps, n=8000)
        if w is None: continue
        results_fw[eps]["nat"].append(jsd(w, rn))
        results_fw[eps]["nnat"].append(jsd(w, rnn))
    if (i+1) % 50 == 0:
        print(f"  ...{i+1}/200 cues done")

print(f"\nFREQUENCY-WEIGHTED NOISE")
print(f"{'eps':>5} {'nat_mean':>9} {'nat_med':>8} {'nnat_mean':>10} {'nnat_med':>9}")
for eps in EPS_GRID:
    nm = np.mean(results_fw[eps]["nat"]);  nmd = np.median(results_fw[eps]["nat"])
    xm = np.mean(results_fw[eps]["nnat"]); xmd = np.median(results_fw[eps]["nnat"])
    print(f"{eps:5.1f} {nm:9.4f} {nmd:8.4f} {xm:10.4f} {xmd:9.4f}")

# explicitly report the best eps per group (the number that decides Phase 4)
best_nat  = min(EPS_GRID, key=lambda e: np.mean(results_fw[e]["nat"]))
best_nnat = min(EPS_GRID, key=lambda e: np.mean(results_fw[e]["nnat"]))
print(f"\nbest eps  — native: {best_nat}  | non-native: {best_nnat}")

  ...50/200 cues done
  ...100/200 cues done
  ...150/200 cues done
  ...200/200 cues done

FREQUENCY-WEIGHTED NOISE
  eps  nat_mean  nat_med  nnat_mean  nnat_med
  0.0    0.3503   0.3399     0.4942    0.4825
  0.1    0.3815   0.3698     0.5165    0.5071
  0.2    0.4151   0.4039     0.5413    0.5323
  0.3    0.4522   0.4433     0.5680    0.5604
  0.4    0.4926   0.4830     0.5977    0.5890
  0.5    0.5384   0.5306     0.6318    0.6239
  0.7    0.6508   0.6461     0.7161    0.7081

best eps  — native: 0.0  | non-native: 0.0


In [30]:
import networkx as nx
import re

def preprocess(df_tokens):
    """Phase-1 rules: lowercase, drop single-char, multi-word, non-alpha."""
    d = df_tokens.copy()
    d["cue"] = d["cue"].astype(str).str.lower().str.strip()
    d["response"] = d["response"].astype(str).str.lower().str.strip()
    ok = (
        (d["response"].str.len() > 1) &
        (~d["response"].str.contains(r"\s")) &
        (d["response"].str.fullmatch(r"[a-z]+")) &
        (d["cue"].str.len() > 1) &
        (~d["cue"].str.contains(r"\s")) &
        (d["cue"].str.fullmatch(r"[a-z]+"))
    )
    return d[ok]

def build_graph(df_tokens, min_count=2):
    d = preprocess(df_tokens)
    edges = d.groupby(["cue", "response"]).size().rename("raw").reset_index()
    edges = edges[edges["raw"] >= min_count]                      # drop count<2
    edges["strength"] = edges["raw"] / edges.groupby("cue")["raw"].transform("sum")
    G2 = nx.DiGraph()
    for cue, resp, _, strength in edges.itertuples(index=False):
        G2.add_edge(cue, resp, **{"R123.Strength": strength})
    return G2

G_nat  = build_graph(nat_sub_long)
G_nnat = build_graph(nnat_long)

for name, g in [("native-built", G_nat), ("non-native-built", G_nnat)]:
    print(f"{name:18s} nodes: {g.number_of_nodes():6d}  edges: {g.number_of_edges():7d}")

native-built       nodes:  16316  edges:   93337
non-native-built   nodes:  14061  edges:   77069


In [31]:
import numpy as np
from collections import Counter

def make_walker(graph):
    nodes = np.array(list(graph.nodes()))
    nbr_cache = {}
    def nbrs_of(cue):
        if cue not in nbr_cache:
            ns, ws = [], []
            for _, t, dta in graph.out_edges(cue, data=True):
                ns.append(t); ws.append(dta["R123.Strength"])
            if ns:
                ws = np.array(ws); nbr_cache[cue] = (np.array(ns), ws/ws.sum())
            else:
                nbr_cache[cue] = (None, None)
        return nbr_cache[cue]
    def walk(cue, n=8000):           # eps=0: pure clean walk
        ns, ps = nbrs_of(cue)
        if ns is None: return None
        c = Counter(np.random.default_rng().choice(ns, p=ps) for _ in range(n))
        tot = sum(c.values())
        return {w: v/tot for w, v in c.items()}
    return walk, set(graph.nodes())

walk_nat,  nodes_nat  = make_walker(G_nat)
walk_nnat, nodes_nnat = make_walker(G_nnat)

cells = {"A_natG_natD": [], "B_natG_nnatD": [], "C_nnatG_natD": [], "D_nnatG_nnatD": []}

for cue in SAMPLE:
    rn  = real_dist_fast(cue, nat_by_cue)
    rnn = real_dist_fast(cue, nnat_by_cue)
    if rn is None or rnn is None: continue
    if cue in nodes_nat and G_nat.out_degree(cue) > 0:
        w = walk_nat(cue)
        if w: cells["A_natG_natD"].append(jsd(w, rn)); cells["B_natG_nnatD"].append(jsd(w, rnn))
    if cue in nodes_nnat and G_nnat.out_degree(cue) > 0:
        w = walk_nnat(cue)
        if w: cells["C_nnatG_natD"].append(jsd(w, rn)); cells["D_nnatG_nnatD"].append(jsd(w, rnn))

print("mean JSD (lower = better fit):\n")
print(f"{'':18s} {'native data':>12} {'non-native data':>16}")
print(f"{'native graph':18s} {np.mean(cells['A_natG_natD']):12.4f} {np.mean(cells['B_natG_nnatD']):16.4f}")
print(f"{'non-native graph':18s} {np.mean(cells['C_nnatG_natD']):12.4f} {np.mean(cells['D_nnatG_nnatD']):16.4f}")

mean JSD (lower = better fit):

                    native data  non-native data
native graph             0.3675           0.6591
non-native graph         0.6224           0.4861
